# Module 03 — Explorer l'écriture bronze MinIO

**Objectif** : comprendre *ligne par ligne* comment passer de l'ingest RSS (module 02) à l'écriture immuable dans MinIO, avant de factoriser dans `src/presslake/storage/` et `ingest/bronze.py`.

**Prérequis**
- Module 02 *done* (`presslake poll` avec dédup)
- `docker compose up -d` — MinIO healthy sur `:9000`
- `.env` avec `MINIO_ROOT_USER`, `MINIO_ROOT_PASSWORD`

**Dépendances** (à installer avant de lancer) :
```bash
uv add boto3 python-dotenv
uv sync
```

Ouvre ce notebook depuis la racine du repo (`data_project/`).

## Carte — module 02 → module 03

```
MODULE 02 (existant)                 MODULE 03 (nouveau)
────────────────────                 ───────────────────
feeds.py                             storage/s3.py      → client boto3
poll.py (fetch, parse, dédup)   +    bronze.py          → hash, chemin, JSON, put
seen.py                              poll.py (modifié)  → appelle put_bronze
data/seen.json                       MinIO presslake/bronze/…
```

**Règle bronze** : on **ajoute** des objets immuables. On ne « met pas à jour » le contenu historique.

## Étape 0 — Imports, chemins, chargement .env

On réutilise la logique du notebook 02 pour les feeds, et on ajoute boto3 pour S3/MinIO.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import boto3
import feedparser
import httpx
import yaml
from botocore.client import Config
from dotenv import load_dotenv

# Racine projet
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FEEDS_PATH = ROOT / "config" / "feeds.yml"
USER_AGENT = "PressLake/0.1 (learning; local dev)"

# Charge .env à la racine (MINIO_ROOT_USER, MINIO_ROOT_PASSWORD, etc.)
load_dotenv(ROOT / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_BUCKET = os.getenv("MINIO_BUCKET")

print("Racine       :", ROOT)
print("Endpoint     :", MINIO_ENDPOINT)
print("Bucket       :", MINIO_BUCKET)
print("User défini  :", bool(os.getenv("MINIO_ROOT_USER")))

Racine       : /home/anthony-marais/Documents/data_project
Endpoint     : http://localhost:9000
Bucket       : presslake
User défini  : True


## Étape 1 — Charger un flux (rappel module 02)

On reprend `Feed`, `load_feeds`, `fetch_feed`, `item_key` — la base ne change pas.

In [2]:
@dataclass(frozen=True)
class Feed:
    id: str
    name: str
    url: str
    lang: str
    category: str


def load_feeds(path: Path = FEEDS_PATH) -> list[Feed]:
    raw = yaml.safe_load(path.read_text(encoding="utf-8"))
    return [Feed(**row) for row in raw["feeds"]]


def fetch_feed(url: str) -> feedparser.FeedParserDict:
    response = httpx.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=30.0,
        follow_redirects=True,
    )
    response.raise_for_status()
    return feedparser.parse(response.text)


def item_key(entry: dict) -> str:
    for field in ("id", "guid", "link"):
        value = entry.get(field)
        if value:
            return str(value).strip()
    raise ValueError("item sans clé")


feeds = load_feeds()
test_feed = feeds[0]
parsed = fetch_feed(test_feed.url)
entry = parsed.entries[0]

print("Feed test :", test_feed.id)
print("Titre     :", entry.get("title"))
print("item_key  :", item_key(entry))

Feed test : france24
Titre     : Fifa : l'étau se resserre autour d'Infantino
item_key  : ce21a36c-a53f-11f1-ad3e-2738758cd497


## Étape 2 — Client S3 vers MinIO (`storage/s3.py`)

MinIO implémente l'API S3. boto3 sans `endpoint_url` viserait AWS — d'où la config locale.

| Paramètre | Rôle |
|---|---|
| `endpoint_url` | `http://localhost:9000` |
| `aws_access_key_id` | = `MINIO_ROOT_USER` |
| `aws_secret_access_key` | = `MINIO_ROOT_PASSWORD` |
| `signature_version=s3v4` | requis par MinIO récent |

In [3]:
def get_s3_client():
    """Équivalent futur de src/presslake/storage/s3.py"""
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=os.environ["MINIO_ROOT_USER"],
        aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",  # boto3 l'exige ; MinIO l'ignore
    )


s3 = get_s3_client()

# Test connexion : liste les buckets
buckets = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
print("Buckets visibles :", buckets)
assert MINIO_BUCKET in buckets, f"Bucket {MINIO_BUCKET} absent — lance docker compose up"

Buckets visibles : ['presslake']


## Étape 3 — `content_hash` : nom de fichier stable

Même `item_key` → même hash → même chemin S3 → **idempotent**.

On utilise SHA-256 en hex (64 caractères).

In [4]:
def content_hash(item_key_value: str) -> str:
    # encode utf-8 → bytes → sha256 → hex string
    return hashlib.sha256(item_key_value.encode("utf-8")).hexdigest()


key = item_key(entry)
h = content_hash(key)

print("item_key     :", key[:60], "...")
print("content_hash :", h)
print("Même entrée  :", content_hash(key) == h)  # toujours True

item_key     : ce21a36c-a53f-11f1-ad3e-2738758cd497 ...
content_hash : bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3
Même entrée  : True


## Étape 4 — Partition `dt=` et chemin S3 complet

Convention PressLake :

```
bronze/source={feed_id}/dt={YYYY-MM-DD}/{content_hash}.json
```

- `source=` : permet de filtrer par flux
- `dt=` : permet de lister par jour (backfill, Spark plus tard)
- `{content_hash}.json` : nom unique et déterministe

In [5]:
def partition_date(entry: dict) -> str:
    """dt=YYYY-MM-DD — published si dispo, sinon aujourd'hui UTC."""
    published = entry.get("published_parsed")  # struct time de feedparser
    if published:
        return datetime(*published[:6], tzinfo=timezone.utc).strftime("%Y-%m-%d")
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")


def bronze_s3_key(feed: Feed, entry: dict) -> str:
    """Clé objet (chemin dans le bucket), sans s3://."""
    h = content_hash(item_key(entry))
    dt = partition_date(entry)
    return f"bronze/source={feed.id}/dt={dt}/{h}.json"


def bronze_s3_uri(bucket: str, key: str) -> str:
    return f"s3://{bucket}/{key}"


s3_key = bronze_s3_key(test_feed, entry)
s3_uri = bronze_s3_uri(MINIO_BUCKET, s3_key)

print("Clé S3 :", s3_key)
print("URI    :", s3_uri)

Clé S3 : bronze/source=france24/dt=2026-08-31/bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3.json
URI    : s3://presslake/bronze/source=france24/dt=2026-08-31/bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3.json


## Étape 5 — Enveloppe bronze JSON

On ne stocke pas « juste le XML » : on enveloppe des métadonnées + un sous-ensemble `raw` sérialisable.

⚠️ feedparser peut contenir des objets non-JSON : on filtre les champs connus.

In [6]:
def entry_to_raw(entry: dict) -> dict:
    """Champs feedparser sûrs pour json.dumps."""
    keys = (
        "title", "link", "id", "guid", "summary", "published",
        "updated", "author", "tags",
    )
    return {k: entry[k] for k in keys if k in entry}


def build_bronze_envelope(feed: Feed, entry: dict) -> dict:
    """Équivalent futur de ingest/bronze.py"""
    key = item_key(entry)
    return {
        "schema_version": 1,
        "feed_id": feed.id,
        "item_key": key,
        "content_hash": content_hash(key),
        "title": entry.get("title"),
        "link": entry.get("link"),
        "published": entry.get("published"),
        "fetched_at": datetime.now(timezone.utc).isoformat(),
        "source": "rss_item",
        "raw": entry_to_raw(entry),
    }


envelope = build_bronze_envelope(test_feed, entry)
print(json.dumps(envelope, indent=2, ensure_ascii=False)[:800], "\n...")

{
  "schema_version": 1,
  "feed_id": "france24",
  "item_key": "ce21a36c-a53f-11f1-ad3e-2738758cd497",
  "content_hash": "bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3",
  "title": "Fifa : l'étau se resserre autour d'Infantino",
  "link": "https://www.france24.com/fr/vid%C3%A9o/20260831-fifa-l-%C3%A9tau-se-resserre-autour-d-infantino",
  "published": "Mon, 31 Aug 2026 13:44:55 GMT",
  "fetched_at": "2026-08-31T13:47:46.928121+00:00",
  "source": "rss_item",
  "raw": {
    "title": "Fifa : l'étau se resserre autour d'Infantino",
    "link": "https://www.france24.com/fr/vid%C3%A9o/20260831-fifa-l-%C3%A9tau-se-resserre-autour-d-infantino",
    "id": "ce21a36c-a53f-11f1-ad3e-2738758cd497",
    "guid": "ce21a36c-a53f-11f1-ad3e-2738758cd497",
    "summary": "L'ancienne légend 
...


## Étape 6 — `put_object` : écrire dans MinIO

C'est l'équivalent d'un `PUT` HTTP vers S3. Le corps est le JSON encodé en UTF-8.

In [7]:
def put_bronze(s3_client, bucket: str, key: str, envelope: dict) -> str:
    body = json.dumps(envelope, ensure_ascii=False, indent=2).encode("utf-8")
    s3_client.put_object(
        Bucket=bucket,
        Key=key,
        Body=body,
        ContentType="application/json",
    )
    return bronze_s3_uri(bucket, key)


uri = put_bronze(s3, MINIO_BUCKET, s3_key, envelope)
print("Écrit :", uri)

Écrit : s3://presslake/bronze/source=france24/dt=2026-08-31/bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3.json


## Étape 7 — Relire l'objet (vérification)

On prouve que le bronze est lisible — base du rejeu futur.

In [8]:
obj = s3.get_object(Bucket=MINIO_BUCKET, Key=s3_key)
downloaded = json.loads(obj["Body"].read().decode("utf-8"))

print("schema_version :", downloaded["schema_version"])
print("feed_id        :", downloaded["feed_id"])
print("title          :", downloaded["title"])
print("content_hash OK:", downloaded["content_hash"] == envelope["content_hash"])

schema_version : 1
feed_id        : france24
title          : Fifa : l'étau se resserre autour d'Infantino
content_hash OK: True


## Étape 8 — Lister les objets bronze

Préfixe `bronze/` = tout le lake bronze. Tu peux affiner avec `bronze/source=france24/`.

In [9]:
resp = s3.list_objects_v2(Bucket=MINIO_BUCKET, Prefix="bronze/", MaxKeys=10)
contents = resp.get("Contents", [])

print(f"{len(contents)} objet(s) (max 10 affichés)")
for obj_meta in contents:
    print(f"  - {obj_meta['Key']}  ({obj_meta['Size']} bytes)")

1 objet(s) (max 10 affichés)
  - bronze/source=france24/dt=2026-08-31/bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3.json  (1109 bytes)


## Étape 9 — Boucle complète : poll + bronze pour un flux

Simule ce que `poll_feed()` fera après modification : pour chaque entry, construire + écrire.

En prod tu garderas `seen.json` pour ne pas re-put inutilement (même si la clé S3 est idempotente).

In [10]:
def ingest_feed_to_bronze(s3_client, feed: Feed, *, limit: int = 3) -> list[str]:
    """Poll un flux et écrit les N premiers items en bronze."""
    parsed = fetch_feed(feed.url)
    uris: list[str] = []

    for entry in parsed.entries[:limit]:
        env = build_bronze_envelope(feed, entry)
        key = bronze_s3_key(feed, entry)
        uri = put_bronze(s3_client, MINIO_BUCKET, key, env)
        uris.append(uri)
        print(f"[BRONZE] {feed.id} | {env['title'][:50]}…")
        print(f"         {uri}")

    return uris


ingest_feed_to_bronze(s3, test_feed, limit=2)

[BRONZE] france24 | Fifa : l'étau se resserre autour d'Infantino…
         s3://presslake/bronze/source=france24/dt=2026-08-31/bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3.json
[BRONZE] france24 | 🔴 Football : Bradley Barcola quitte le PSG pour Li…
         s3://presslake/bronze/source=france24/dt=2026-08-31/2a9785e96b86602fb3f428ee388fd7e857f6d86f450fa8bd9dd8d9193ec37d3f.json


['s3://presslake/bronze/source=france24/dt=2026-08-31/bb895855b9cc591f905621d627b32272ba025ed88f1d2cbc68ec4bd7ba0c02a3.json',
 's3://presslake/bronze/source=france24/dt=2026-08-31/2a9785e96b86602fb3f428ee388fd7e857f6d86f450fa8bd9dd8d9193ec37d3f.json']

## Étape 10 — Idempotence : même item → même clé S3

Réécrire le même objet ne change pas le chemin. C'est voulu.

In [11]:
key_a = bronze_s3_key(test_feed, entry)
key_b = bronze_s3_key(test_feed, entry)

print("Clés identiques :", key_a == key_b)
print("→ 2e put_object écrase le même fichier (contenu équivalent si même fetch)")

Clés identiques : True
→ 2e put_object écrase le même fichier (contenu équivalent si même fetch)


## Étape 11 — Où ça se branche dans `poll.py` ?

```python
# Dans poll_feed(), après mark_seen → False (item nouveau) :

envelope = build_bronze_envelope(feed, entry)
key = bronze_s3_key(feed, entry)
uri = put_bronze(s3_client, bucket, key, envelope)
print(f"[NEW] {feed.id} | {title} | {uri}")
```

Le client S3 est créé **une fois** par run dans `poll_all_dedup()`, pas par item.

## Étape 12 — Refactorisation vers le package

| Notebook (ici) | Fichier prod |
|---|---|
| `get_s3_client()` | `src/presslake/storage/s3.py` |
| `content_hash`, `bronze_s3_key`, `build_bronze_envelope`, `put_bronze` | `src/presslake/ingest/bronze.py` |
| branchement dans la boucle | `src/presslake/ingest/poll.py` |

Tuto détaillé : [`docs/modules/03-minio-bronze.md`](../docs/modules/03-minio-bronze.md)

## Critère *done* module 03

- [ ] Notebook exécuté jusqu'à l'étape 9 sans erreur
- [ ] Objets visibles dans la console MinIO sous `bronze/source=…/`
- [ ] Code factorisé dans `storage/s3.py` + `ingest/bronze.py` + `poll.py`
- [ ] `uv run presslake poll` écrit des `s3_uri` dans les logs
- [ ] 2ᵉ poll → 0 nouveau

**Module 04** : catalogue Postgres (`url`, `s3_uri`, `content_hash`, `status`).